# 8. Redes Generativas Antagónicas (GANs)

En este notebook aprenderás los fundamentos de las GANs, cómo entrenar un generador y un discriminador, y verás ejemplos prácticos con TensorFlow/Keras.

## Objetivo
- Comprender la teoría y arquitectura de las GANs.
- Implementar y entrenar una GAN para generación de dígitos.
- Analizar el impacto de los hiperparámetros y visualizar resultados.
- Aplicar buenas prácticas para entrenamiento estable.

## Prerequisitos

> 📌 **Prerequisitos:** Haber completado los notebooks [04 (MLP)](./04_redes_neuronales_capa_densa.ipynb) y [05 (CNN)](./05_redes_convolucionales_cnn.ipynb).

- Conceptos de redes neuronales, funciones de activación y entrenamiento con Keras.

## 1. Introducción teórica

Las GANs consisten en dos redes que compiten entre sí:

- **Generador:** Crea datos falsos a partir de ruido aleatorio.
- **Discriminador:** Intenta distinguir entre datos reales y generados.

El entrenamiento es un «juego» donde ambas redes mejoran mutuamente.

**Ventajas:** Capaces de generar datos hiperrealistas (imágenes, audio).  
**Desventajas:** Difíciles de entrenar (inestabilidad, mode collapse).

## 2. Importación de librerías

In [ ]:
# === Reproducibilidad ===
import random
import numpy as np
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
tf.random.set_seed(SEED)

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

## 3. Carga de datos y definición de la GAN

Crearemos una GAN para generar dígitos similares a MNIST. El generador usa **BatchNormalization** para mejorar la estabilidad.

In [ ]:
# Cargar MNIST
(X_train, _), (_, _) = keras.datasets.mnist.load_data()
X_train = X_train.astype('float32') / 255.0
X_train = X_train.reshape(-1, 28*28)

LATENT_DIM = 100

# Generador con BatchNormalization (buena práctica)
def build_generator(latent_dim=LATENT_DIM):
    model = keras.Sequential([
        keras.layers.Dense(128, input_dim=latent_dim),
        keras.layers.BatchNormalization(),
        keras.layers.LeakyReLU(0.2),
        keras.layers.Dense(256),
        keras.layers.BatchNormalization(),
        keras.layers.LeakyReLU(0.2),
        keras.layers.Dense(28*28, activation='sigmoid')
    ])
    return model

# Discriminador con LeakyReLU
def build_discriminator():
    model = keras.Sequential([
        keras.layers.Dense(256, input_dim=28*28),
        keras.layers.LeakyReLU(0.2),
        keras.layers.Dense(128),
        keras.layers.LeakyReLU(0.2),
        keras.layers.Dense(1, activation='sigmoid')
    ])
    return model

generator = build_generator()
discriminator = build_discriminator()
discriminator.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

discriminator.trainable = False

# Modelo GAN combinado
z = keras.Input(shape=(LATENT_DIM,))
img = generator(z)
valid = discriminator(img)
gan = keras.Model(z, valid)
gan.compile(optimizer='adam', loss='binary_crossentropy')

print('Generador:')
generator.summary()
print('\nDiscriminador:')
discriminator.summary()

## 4. Entrenamiento de la GAN

Entrenamos alternando discriminador y generador. Usamos **label smoothing** (etiquetas reales = 0.9 en lugar de 1.0) para estabilizar.

In [ ]:
def sample_images(epoch, n=10):
    noise = np.random.normal(0, 1, (n, LATENT_DIM))
    gen_imgs = generator.predict(noise, verbose=0)
    gen_imgs = gen_imgs.reshape(n, 28, 28)
    plt.figure(figsize=(n, 2))
    for i in range(n):
        plt.subplot(1, n, i+1)
        plt.imshow(gen_imgs[i], cmap='gray')
        plt.axis('off')
    plt.suptitle(f'Época {epoch}')
    plt.show()

def train_gan(epochs=500, batch_size=128, sample_interval=100):
    half_batch = batch_size // 2
    d_losses, g_losses = [], []
    
    for epoch in range(epochs):
        # Entrenar discriminador
        idx = np.random.randint(0, X_train.shape[0], half_batch)
        real_imgs = X_train[idx]
        noise = np.random.normal(0, 1, (half_batch, LATENT_DIM))
        gen_imgs = generator.predict(noise, verbose=0)
        
        # Label smoothing: 0.9 en vez de 1.0 para reales
        d_loss_real = discriminator.train_on_batch(real_imgs, np.ones((half_batch, 1)) * 0.9)
        d_loss_fake = discriminator.train_on_batch(gen_imgs, np.zeros((half_batch, 1)))
        d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)
        
        # Entrenar generador
        noise = np.random.normal(0, 1, (batch_size, LATENT_DIM))
        valid_y = np.ones((batch_size, 1))
        g_loss = gan.train_on_batch(noise, valid_y)
        
        d_losses.append(d_loss[0])
        g_losses.append(g_loss)
        
        if epoch % sample_interval == 0:
            print(f"Epoch {epoch} [D loss: {d_loss[0]:.4f}, acc.: {100*d_loss[1]:.1f}%] [G loss: {g_loss:.4f}]")
            sample_images(epoch)
    
    return d_losses, g_losses

# Entrenar la GAN (reducido para ejecución rápida)
d_losses, g_losses = train_gan(epochs=500, batch_size=128, sample_interval=100)

### Evolución de las pérdidas

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(d_losses, label='Discriminador', alpha=0.7)
plt.plot(g_losses, label='Generador', alpha=0.7)
plt.title('Evolución de las pérdidas durante el entrenamiento')
plt.xlabel('Época')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

## 5. Resultado final del generador

In [ ]:
# Generar imágenes finales
sample_images('Final', n=10)

### Recomendaciones prácticas para GANs

| Aspecto | Recomendación |
|---------|---------------|
| **Estabilidad** | Usa BatchNormalization en el generador |
| **Label smoothing** | Etiquetas reales en [0.8, 1.0] para el discriminador |
| **Activación** | LeakyReLU en el discriminador, ReLU o LeakyReLU en generador |
| **Mode collapse** | Prueba diferentes learning rates o usa WGAN |
| **Evaluación** | Monitorea calidad visual, no solo la pérdida |
| **GPU** | Esencial para entrenamiento razonable |

## 6. Discusión y Conclusiones

- Implementamos una GAN funcional con BatchNormalization y label smoothing.
- Las GANs son difíciles de entrenar; monitorear visualmente es esencial.
- La calidad mejora con más epochs, mejor arquitectura y técnicas de estabilización.
- En el siguiente notebook veremos Autoencoders, otra familia de modelos generativos.

## 7. Ejercicios Propuestos

1. **Ejercicio 1:** Entrena con más epochs (2000-5000). ¿Mejora la calidad visual de los dígitos generados?

2. **Ejercicio 2:** Cambia el dataset a Fashion MNIST. ¿La GAN puede generar prendas de ropa reconocibles?

3. **Ejercicio 3:** Implementa una DCGAN (GAN con capas convolucionales) para mejor calidad de imagen.

4. **Ejercicio 4 (Avanzado):** Implementa una Conditional GAN (cGAN) que permita generar un dígito específico pasando la clase como condición.

## 8. Referencias y Recursos

- [TensorFlow DCGAN Tutorial](https://www.tensorflow.org/tutorials/generative/dcgan)
- Goodfellow et al. (2014). *Generative Adversarial Nets.*
- [GAN Lab (Interactivo)](https://poloclub.github.io/ganlab/)
- Géron, A. (2019). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow.*

---

📎 **Notebook anterior:** [07. Transformers y Atención](./07_transformers.ipynb)  
📎 **Notebook siguiente:** [09. Autoencoders](./09_autoencoders.ipynb)